# Research Skills FSS 2026: **Openness in and with AI**
## *"Start small, start open, validate"*

### Topics

1. Using open-weight / open-source models with Ollama
2. Challanges regarding reproducibility / replicability
3. Controlling the randomness of model outputs
4. Running some tests with synthetic data

We follow in parts an approach that was first highlighted in this paper:
>Stoltz, D. S., Taylor, M. A., & Kumar, S. (2026). Selecting Language Models for Social Science: Start Small, Start Open, and Validate (arXiv:2601.10926; Version 1). arXiv. https://doi.org/10.48550/arXiv.2601.10926

In [31]:
# Install dependencies
%pip install "litellm>1.83.6" json_repair

Note: you may need to restart the kernel to use updated packages.


In [32]:
import json
from json_repair import json_repair
from litellm import completion

### Starting small and open with `smollm2:1.7b`

#### Task: Sentiment analysis for social media posts

LLMs can and are used for a variety of data processing steps in various scientific domains. In some areas they show advantages in comparison to traditional systems. See for example the following paper evaluation sentiment analysis for historical domains:

> Klähn, J., Burghardt, M., & Borst, J. (2024). Death of the Dictionary? – The Rise of Zero-Shot Sentiment Classification. Proceedings of the Computational Humanities Research Conference 2023, 3558, 303–319.

We run a hypothetical sentiment analysis task on synthetic digital behavorial data (social media posts). **We start small and open** with [smollm2:1.7b](https://huggingface.co/HuggingFaceTB/SmolLM2-1.7B) that is available in the [Ollama model library](https://ollama.com/library/smollm2) and [licensed under Apache 2.0](https://huggingface.co/HuggingFaceTB/SmolLM2-1.7B#license).

`smollm2:1.7b` is one of the higher ranking open weight models on the [European Open Source AI Index](https://osai-index.eu/model/smollm).

![Smol-Default](img/osai-smol-02.png)
![Smol-Chart](img/osai-smol-01.png)

In [33]:
def run_completion(
    model: str,
    prompt: str,
    temperature: float = 1,
    top_p: float = 0.5,
    seed: int = None,
    iterations: int = 1,
    max_tokens: int = 100,
    verbose: bool = True,
    evaluate: bool = False,
) -> str:
    """
    Helper function to run liteLLM completions and clean the
    response output.
    """
    results = []
    for _ in range(0, iterations):
        response = completion(
            model=model,
            messages=[{"role": "user", "content": prompt}],
            api_base="http://localhost:11434",
            temperature=temperature,
            top_p=top_p,
            seed=seed,
            max_tokens=max_tokens
        )
        result = response.choices[0].message.content
        result = result.strip()
        
        if verbose:
            print(f"{result.strip()}\n---")
        
        if evaluate and result not in results:
            print(result)
            results.append(result.strip())
        
        # Clean markdown json strings; transform str -> dict
        if isinstance(result, str) and result.startswith("```"):
            result = json_repair.repair_json(result)
            result = json.loads(result)
        
        results.append(result)
    return results

The next cell defines the prompt with the labeling task and the social media post as "sentence".

In [34]:
prompt = """Categorize the following sentence with exactly one of these labels: positive, neutral, negative.
Return ONLY the label.

# Sentence
Big policy announcement today. Bold move. Let's see how long before the exceptions quietly swallow the whole thing."""

In [35]:
results = run_completion(
    "ollama/smollm2:1.7b",  # The model we are using
    prompt,                 # Our prompt
    iterations=10           # Run the same prompt with the same model 10 times
)

negative
---
positive
---
negative
---
negative
---
negative
---
negative
---
negative
---
negative
---
negative
---
negative
---


After running the same model for 10 iterations, we can see changing model outputs. This variability highlights a critical risk in using stochastic LLMs for academic research: the lack of deterministic responses creates **reproducibility challenges**.

We can control the probabilistic nature of language models by refining some **parameters** in our `run_completion()` call:

In [36]:
# Step 1: Lowering the temperature
results = run_completion(
    "ollama/smollm2:1.7b",
    prompt,
    iterations=50,      # Run 50 times
    temperature=0,      # Temperature controls the randomness of the model's output; lower = less random
    verbose=False,      # Disable printing the label for every run
    evaluate=True       # Only print distinct labels
)

negative


In [26]:
# Step 2: Passing a seed
results = run_completion(
    "ollama/smollm2:1.7b",
    prompt,
    iterations=100,     # Run 100 times
    temperature=0,
    seed=42,            # Fixed seed for the random number generator to make outputs deterministic across runs
    verbose=False,
    evaluate=True
)

negative


### Proceeding small and open with `yulan-mini-instruct-v1` and `olmo-3:7b-instruct`

We now test two other small and open models ranking high on the [European Open Source AI Index](https://osai-index.eu/):

- `yulan-mini-instruct-v1` (2.4B)

![Yulan-Mini](img/osai-yulan-mini.png)

- `olmo-3:7b-instruct`

![Yulan-Mini](img/osai-olmo.png)

#### Task: Named entity recognition (NER) for historical newspaper articles

We define a named entity recognition task for a paragraph of the German newspaper "Deutscher Reichsanzeiger und Preußischer Staatsanzeiger" (*German Imperial Gazette and Prussian Official Gazette*), which was published under changing names from 1819 to 1945 (https://digi.bib.uni-mannheim.de/periodika/reichsanzeiger/ausgaben)

In [27]:
ner_prompt="""Perform the following tasks for the given sentence: 
1. Annotate the sentence for 3 entity types: PER (person), LOC (location) and DATE (date).
2. STRICTLY adhere to this JSON response format: 
```json
{entity 1: "entity type", entity 2: "entity type" ...}
```
3. Return ONLY the given response structure and nothing else

# Sentence
"Her Britannic Majesty and the President of the United States, having concluded a Convention on the 13th. of May 1870 whereby British subjects who have become and are naturalized as Ctizens of the United States are at liberty to renounce their naturalization and to resume their British nationality provided that such naturalization be publicly declared before two years after the 12th of May 1872, and Citizens of the United States who have become and are naturalized as British subjects are at liberty to renounce their naturalization and resume their nationality as Citizens of the United States provided that such renunciation be publicly declared within two years after the exchange of the ratifications of the Convention, concluded a Supplementary Convention on the 23d. of February 1871 prescribing the manner and form in which the renunciation by the subjects of the Contracting Parties and the resumption of their native allegiance may be made and publicly declared."
"""

In [28]:
results = run_completion("ollama/yulan-mini-instruct-v1:latest", ner_prompt, iterations=3)

### Analysis:
- **PER**: Queen Victoria (Her Britanni c Majesty)
- **LOC**: Convention held on May 13th, 1870 in Washington D.C. (United States)
- **DATE**: The convention took place on the 13th of May 1870

### JSON Response:
```json
{
  "PER": "Queen Victoria",
  "LOC": "Washington D.C
---
```json
{PER: "Her Britanniic Majesty", PER: "the President of the United States", DATE: "13th. of May 1870", LOC: ""}
```
This response adheres to the given JSON format and only includes the required entity types with their corresponding values from the sentence. The location (LOC) is not present in the sentence, so it's left empty.
---
```json
{PER: "Her Britanniic Majesty", PER: "the President of the United States", DATE: "13th. of May 1870", LOC: ""}
```
---


#### Validation

Out of the box `yulan-mini-instruct-v1:latest` seems to have problems to align to our JSON output format. In a real-world scenario we would validate such test runs against ground truth data (created by human annotaters for example).

#### Testing with `olmo-3:7b-instruct`

We now test a bigger model (with 7B paramaters more than double the size as `yulan-mini-instruct-v1:latest` with 2.4B parameters)

In [29]:
results = run_completion("ollama/olmo-3:7b-instruct", ner_prompt, iterations=3)

```json
{"Her Britannic Majesty": "PER", "President of the United States": "PER", "13th of May 1870": "DATE", "12th of May 1872": "DATE", "23d of February 1871": "DATE"}
```
---
```json
{"Her Britannic Majesty": "PER", "President of the United States": "PER", "13th of May 1870": "DATE", "12th of May 1872": "DATE", "23d of February 1871": "DATE"}
```
---
```json
{"Her Britannic Majesty": "PER", "President of the United States": "PER", "13th of May 1870": "DATE", "12th of May 1872": "DATE", "23d of February 1871": "DATE"}
```
---


In [30]:
results = run_completion("ollama/olmo-3:7b-instruct", ner_prompt, iterations=10, verbose=False, evaluate=True)

```json
{"Her Britannic Majesty": "PER", "President of the United States": "PER", "13th of May 1870": "DATE", "12th of May 1872": "DATE", "23d of February 1871": "DATE"}
```


#### Wrap up
`olmo-3:7b-instruct` works well on our test sentence. It is a small and open model that we can download and share as it is [licensed under Apache-2.0](https://huggingface.co/allenai/Olmo-3-7B-Instruct#model-description).